# jitmo tool-picker fine-tune (self-contained)

Same as `finetune_router.ipynb`, but the training data is embedded below (no Drive, no upload) and the finished model is downloaded by the browser. Runtime > Change runtime type > GPU, then Run all.

**Never train on `testset/heldout.txt`.** It is not used here.

In [ ]:
#@title 1. Settings
BASE_MODEL = "unsloth/gemma-3-270m-it"  #@param ["unsloth/gemma-3-270m-it", "unsloth/functiongemma-270m-it", "Qwen/Qwen3.5-0.8B", "unsloth/gemma-3-1b-it"]
EPOCHS = 3  #@param {type:"integer"}
LEARNING_RATE = 2e-4  #@param {type:"number"}
LORA_RANK = 16  #@param {type:"integer"}
DATA = ["/content/data/router_train.jsonl", "/content/data/router_train_args.jsonl"]
OUT_DIR = "/content/out"
NAME = BASE_MODEL.split("/")[-1] + "-router"

In [ ]:
#@title 2. Install and mount Drive
!pip -q install -U transformers peft trl datasets accelerate sentencepiece gguf
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
#@title 2b. The training data (embedded, no upload needed)
import base64, gzip, os
os.makedirs("/content/data", exist_ok=True)
A = "..."
B = "H4sIAAAAAAACCu19SXfbSNblvn/FszfeuHVyUn9l9UIHFmGRJQ76SMgudWcvQmRIRBIEmABoJbNO/fc+EeAAzgQZAyjfTaYtEwEIl2+6776If78f8iRhLzx5f/V///2+G4UpD9P3V++/+EFAfkhpn9N3Fox5Qs9RLP+aRlFAv79vjXhILCQ2Gv3+np7jaCj/dZzw+ENCMf9zzJP0gm6i0UT+w2sU95L5Ryhhfu9/Uy+iMErJD7/zMCUWTtK+H75c0EPCxdp8OEonlKSxH77I+88+sVin5/fkEgmbXPwe/h5+lQ979Xv4P8WTXckPstGI0oiiEQ8/EkuI0YjHSRTSazQOehSyISc/ff/xfRwF/P3V+2SSpHz4/j8f828k5LxHPT/pRrG44+vi4+I5Vj7879/fy/dy9fv76SW/v//P4gqWJH6SsjB9/5//95//8W+AcCgIScrSmMYjGjE/TA+FQH4YAKgAIGDjsNun70H30Jf/Peji1Sv67scpvvv2AIgGxHsvnAb9KIgOff/iCrx+pd//lAf8JWbDQyGYfR4wqIChRt+cpkdeix46LnlO544aTtO5ddsH48GSAQ1ZyF54DEyU5aaj6JXHo8gPU5GeEkvY6OAgMb8UeOioFWgUcJZwlAwWsIjHIXX7cTQ8+PVnn8bbV/H2u/Kzk2gs1xF1g7CHdyge7OKQD8EI27ZA8emVhalYZ5xw1NSWggPKiVJUdYjR1iEQXAV4DbtRIIxSPmK9Q2GYfhxIKCvbXgv2d15RqanyQ6Mo9Z8nxfjV6UVAQFU6hCBgPwhMv9SwgXJVyjQKElTL5aGSUC5biA+LXgG6C1Y1GF0WdMcBS6ODGbzFFQBCWb0gJQEF6gXkTIrtAHIYO6+/zyckAkLC09QPXw7OjGafBwoKjYD/xbvBwS7ory4P8PpVvP76Q/OmSvdOrenBBRl/+xPGYprT12DubNYFf0fRwS008Vm8efUWgJrYOi0hYysCsfnmQUToH5QkGZ0pH6F6tF4WCHKiGAzgJ3QM7KQhygPL7ugCjsgCBkv9s0KRGR00zakqKjb781Puv27cOsoFO3WzoC2mzh6xwc5A7VLN9g5FmyXyDj2c8oi/Fl36QjOcaO7r5fUgi7SlsJhq3I8YpIU6XiEWT33my5TptUC+hB6bniCBoTWbWIyH0hDQbrMelmethrTPwkGCKs4eyzqL0ojQ9hoO06qsYAcUtZzqjHUWbYuokRChtSSruWq6wPQOimktMUJ0Ea7RcrA61/wOpZvlnBVkq2WhWAknCTucxd1+9oL5U0mB+HPM48kVvfaZ5CGSpWcWy+3NT1cvoNEk7UchBX6SUjcajmLe52HiR3vlMvJZMsHMtiWA1NFIBVE0yJRLScoo5l1/xBOKwsAPeQFk8lcDjdPRmKT9MNrwTZ8ic0iGC7sx7eHyRgDbsWU7TzxJKWCjNBol9MtPv/yvwt5sbQWgcjIq/ehVXJ/6nJj8b1FQVhcAJkayM0Qa22jdVF3nnqotz613qNak25ZDHddp31Tpzmm3Dgen2+dMGGLKg0S815eIARaVRhSwVEQOf9SPRIoW+90C/m3DxUDnaHTunY7nUNu9qd27nePMBXmZ2gxgg/spnATAhalE5iWKXgK+lpohJytHOEljJt6LP5Sd3tE45NTjQd8/HJ9tKwCnU21mrUREdWkztmz7ok8DzDuYTLlc24Y4jiTADjpbTWe60oDFEczHIkDLnZkcKteoZYzHm/Ui/Tg7QbWvISkD/1I2RND0LyU6ciY632veTP1Pe80HSFsBlx33dqAGH15Ob/m5CRlhPDwEMnaQqbdad/Qg2mbfxAYcXs0lR/631azXmi4oThs8QEQH8jWoNy3C9NCg21brtu7S/aNXbTWpXut4dNNq3Lfdqtvs1FpNpANWCYGtCdtRpSgg0hB+HM/teFS7r7aaLt23azfucU1oZAUaEumt0eeQXVgQe7S2ClY7BWhOlwCWR8dp0yyl3poVFM6tEXmMNNvWBdGFGB30q9WhEw0IgugzarGhw2YaijWzOAoN2IapnPka+XK5oj0S5vIYDWRO5YkrfEKH5MbIi0sWZALe94sVLIgz+jvQYQSBTSmUaP0RlGjlap5h9KzsA04XyIltic5eoih4QWJcQqUmNi+xicI6R39UMEdNog6S8ZD26ftCpL/lH5ctrJFF/ahD+reHRwY85d2kqbAFIWs22s9HflBG8T+SA1vIbNpp5qh0GpWm6hYMps3Lr1DCFidl3RJ415gmZJe2Z5uwN8C5MwXZ2WKl3S3I6fXk+xSvuaRoCZiyIxzkU1LqpwHP/oGHV9RjE2JhT7wxLg92EI90RVFIz7Ev/zGl30bDj5RGwyiOo1fxg59/2keeChcp7tnjw0g+86f95yWKJ80wnF74+/uP4qc8lD/9pOt8JoC4GUTW68lbdsXYdCpOcwqEvQ8n4o887LGYwmj/xhFLsObWWkZXrAR4TcKbdPu8NxYEoDS3xH8JKZmEXRpGoVj0558Km+x8jWVscwsCYpMQd26qbuWh7pJXdalzX2s3PbqvO81mrXlLXqvRardb38jx6Nf7RiGok1HshymNAhaGfviyDHf+EX8dAXIrTnsFoZMd907E4bytJVhP494LTynm333+Kh/+clTMbS+tsIzrJazXLKyfxURn5qzbtbyz/tJqw2H/IKVSbrUTS6bcSgDVKKjiCNylBDvlbEjinffGo6MRzi8CpEuVby3hu5JsHVtQbYcbFZUluLsxZymfv5qdpi03L7mGgb+R+DyKoz94N6WB3x1Ez8/Z+4tC/6WfioX+UQjolcVW87HFqgDbtnnLvGwlRT4xL0PCXYKInac8VwL2f139+tNIEfmZrQWISwbxaQa9BWvYdXn6GXmnPYxiwXqo8trT5QCv7dC8lpLlku9MOaAuKUMCbjsBX21pHEN27+RBQXfbdNmrxly4RbXTfNGiKl8thbB8/ijftF3Hc8lpkvvVbXqyeVVxO7XbJnUemzd0WbBRtTW3hnMugf0ud5uLavp2dJoh7rMZelczqyP4j52JFTiQUqo3j+1LbRVxoi1VxkT6WJh35tOAumyy3U9s+E5NqoVgbIvi+HPM4pTHwSQv6DuFql5dEHx1OQ0Z8qAfkLw8ka7emXID8TKYdT75/qQqy0Z4tmXEeTiXIvMxTOZWfEFolkKCX3EbrZN199D+lBPc+3brn+6NR3e1m7vWly8SZ8FSFYR3Z4kM1quMpPUR3OYO3hoYl5LZPALkreEYEFtIoiPa3D5e0F3HT0ltYb1QMZXIZ68TYUfCvZsAA+Ylwjzvz7GxyBu35/yMyzHk1z5iGxM0JaC/ljPvU/oYO1JwFNQlyMVXkT41G0fJdR4MKEjtN9mZOlYysrMlBclIiWPytNdwrTwqo4lhdQBS6u03Tz5Cfv+WRSRD7DnyRjCWByVvU/1dqppdxnyFzc05V/uRjVaz4jzSzz85DWh2fzAt/oGHO0GSf57jjxl/8e7EtAssSIkgvRwNL07EE+G3NJnzJzZUtWsbWK3yUB3YF/ms9SDr2/sctUsX4uh5bOVT1Fp3Fj6w1zIxzkdumbmTbEa7v8T932O1WzsQh3CrRJ2FeWPhSMve1FSAQZfuYJEVoQ2F0Ss0PD9IOnZErm1tNqbDU3E8K6Mh56nsSJYSa/myM7CnD/qRnibkp0kGO4nfyh9O4Y/EgZvS4/rhdz/lezD1w24w7nH67g9iNpy9L+Fo92E4NY911xzJH2YLAjhdwCXicxl2o9ifMIXQyfWAnC7kRJt2bna8p9LmeA+w6YKt1vxa81zqOA3yWhvP3rqvd4oBuFkFl2GZwHnqw3I8JFGfTIMeS8TjsjTlYY9zkSUhBp5V8rIUAJeqh2IAbio8EBINACmMMdlhiXMp0nVRi1wVMcG5agdzwlg8/eTMw6bRWve0GJAbOqdwsWZcrMRxyHnMZjCuFOzFkNxc7WdgyrsAS+1YLqBcnY4shuWW2UqAaShkZnnP5qCJNOg8Md1bk1ygKCk1iNGAYJtvt9b8I+qHWktNcQPAaBLGFRpOHYcHKA2ks8slpjYsETa1h82ZbQ7Ykx9rN055F+Cp2Tpj1h8HM+PMjzUdNru0DOf68FIGpbwJoDTqaLXkPnCyZpDkvS1c3rU6Mg/NaIvsj7aoiSrTjBwLupBzgyxrkehkBsComwiOmVvVmeXAh2qPjVnZsVdOoEZNgPLDcNKKkHgeITHk/XlE1NhpFrcBmJr7WVl6Y7jaQMZjQ4muQ6YFd6s9QCZahXaQTFqkc3LEtxrGHCWIkQRoqZOlHER0sIzzAZiuO/O+IxqO54Nh5j3XmbhXroqJgwfVjWPqT7VW+jJTqKyMpzLJCJqc8/WrkpabJzRzTvtCDRsONk4nhE6lQl9rd22nQU6HnCY5nuc2K65LraYcSr+p19ymRzdOva4m14Emx267Kh/3rhWFTOSvRmJmBqtuISvAtDuUBcn5G2Vdtc05gwLSniP9s1VtbsmQKm6ndtukzmPzRk3GiwJU775LN/WHiktOs+Y5VMsw9FynQR3PaVYe7hXlRCz0Uxilzv3qdu/ygp7lWZIJeXJPQXcEvtQ0sa4ANNB2elH7Hq7oO7SVHPCYBjIZsYXkNI/5/FC5dT1qu19r7jdVXS2AqLlolAS64ZoRtLqNbbAVTwqA0bHLpYODPe/I2XSrcwrgvx+ctue2649HRU8MFJRrYFJD9Y/hAe0s63279uiYolnBmpsePV+JeDQKEoXdLYyEmN22Fy72Texvv5y5qJyjhIM1vvXg0nnSLGHl18TeR4l4++RxNkxKCuNT1Jtc5e8d0ShKUvFP3T4LQx5k/yp/h9mPKGTDQwYmUx4E8mo2ZH9H4fxyUWLK9dI+SynhnCbRWJyrdil+Eg72hk7x1Bl/l7tWojq9RdamlHcFtsqxneP66g/8Ee/5bAu0GZrzdzce9VjKDwZ349VrKM+fAUArB3o0/eXFxS9D5gfzi6fwcnr2A06+eA+sNylqvmsLrKEr7wpklSM7fWubrfgKhvvWDVc/vLBc/ZYb8vQ58P9arCj3ix1yemLdwcF45i9ag3F6CwCp1UbXEylpp9PKiobRdEun7Ezig5Hdcj1csSWYV+x1J8jXylGGKevCuOF2Os6tm+ltv9U8z23TTdVpNt36lZwvo4ZLn52bOzU+OX3105THANJ0WpynG9RwFPC8ZjxvkrLuIPrO4+cgel32v/6HIKCEhz3yU0qj0H/ppweDu+niNZCXbg6glQNddR9ppwuufWhQ+yE7errueG4BeIcUj7OWXrCpyoEr1k4yvvhpf/y0hWFUX/BktwOg+gAN/HDAe/62fsCqzakz1tmNAa5ycMdDyudOS5zT1QaiOIxewRKfVTo8icbp+IkvVtRnp9NbAVBrPjgfIQvZ6s7QCv+rDVh5kvwc3VWeaRlcU6wi+CYj7Z3l9NhQSYssWRO0962ON9Pt12vNO7dSa84KWfKqjkcd16XH1gM5nioSCn7ZSAYV817PT3U267I7AEatBH/brVRq3oJcytsj3dc7amwSUBqwyJUy1UDYRKGqt66Z8rGHlDUHjGigHWedVZoju7mDY4o6RAvHXp5kql5FyNXtnQ+x4Y3qwoMGPgoKFGHS5kiKlQEQYyJUjIDY0o+vNnwKGvC+jg90Mwbc9UrLZy3Z0l0toQ9U3oiMYPw2bRo93beWZ+n308iy7CpY19IlhYkWHLPFcS79posevaURTA3CVfQWrIgtttCLhfoLh1KUkF3YzY/VWy3yYq2SuH3Gi/mBN1PnYB+htzniY2SzChitZoh3S5KNNf1Q79hr9+kjKeChDYC6Kq3STBhDVWVZLKd+swNAqgvSJFqqdJYVVGanfqCiMsAUr47vmRPJYVZEv19eMV9TLVpYrm5gd+6DjD7tm3DNK0nV+pz8O+TI5Uf0obG0o9Btw6nVFyNf4kdfanWXah1qu07lEXtZvGkmCiTUmYAcDWinoGJbr7YQ04j9D8o0jntb86oPn/N7vdXr1HGbFap55LWatduqh70P3rL1wnDPcncLp+H8n9bK3hb6jRc1kCZon/rMXzLfjYLVK4hV374eOd8EeKemdQBkTe4ftqxXNXRcFwpfSwcQ6PTIWlMpV35NmXyEkqIqHi0X89JIHLTOun3xb+niVEQufxOZtuxBM/vkcELd7+JCeazoPryy9yMSYHGZxCjVfirpjwuOeIUv4ZCH6UUv6v5VHKaVBQCYNsBktTA3JnncqEgySCxwvFXpO1L0h8Ro9uEwSnlykf6VykVDPz3couaX5mGSSwAntc4vYElKr5wP5K85imKJ1nd/ELPhwXCtL5LHLVsMwGmPWsKPqQha8Icq4XJlL7LxSDdfyWvRnfO51j4+Vg3Ykx8DHKUJxXqgOiKrQMgyWEzFPBkP5YpJgTA1vzAPUYLQpA6g7PK12MRE1j5fQ2B2IBN1SLQCgIaSQhl6FOWECGPqw5hYT/woZt2TIll+HQQz3b5SrOiH3yO/y1f9ZMz642KwTRfKoyYXAWrKLS0J/B6XtL8QJxxnZdkaS7Uy7wErxRa2IZqtGBoiW/ntbeYjZ+z8wuQuTnWRIOnN8FMyFF2oyPkR1DQgJhZ8GvdeeEpJn/MjU5D8CoDMRPY4zUROimmbsxHEMt2545EtzM1wgbc3EsZC3leiDRDrAC5z5XQh3LakisBMfWtMiMtrza+t2o0rGmT37dqjcypOo9ifAChNuhuZyqkQ3iAn1OECM5zWey7H44QWi44scLXWWqE1lFRdoDaUYud4nnNTlRGrU69V3A45zQrJKCYHoOifrWoTWfx58YegD89QQ1BQobijDIM8UbekSug9FAmqkImYYDiKibV32BYkA8bUOYXEv7tbmMhAzMczSVgccgbFIWYH9sNMxbaisLpWUrLBZ2rqbabsL4p5Oo7Do9jgxeUghK0rdhJl00nIKLUY22q5jSL7nIq0ovJ81GpWFN4i+CgUeCOWadz5oZiSCq0yY2qBXFp4sv5+c44I7JRiNx5SDr6lmuxUxce2mgyeUXks285iHRHRwGfZ7ahJoai6jhp0p8Z0IcVkwtiOwIBtbWjIHNdBw9C0afua7ktwUh65cZcCpJD6eariExMIXbYc4nEzt7tUIBi8VS8XXmzTkolsjtMMr3tDSHbMBrDsfV8DtLNQ73znPXjEUg9QZGIPBQMUEHro8IOLvtiKH8QmpucprsLGmGcz/dd0qw6GNM+FPSyowl+LX8gJzYjdCuK0vZUJwMxmHIU6mBsTDjhDjaIO2a4Mo9cynj3wxRfU2RmhI87GoSimV3m6dkp+su8Y5zlfMWB9FlKf+cdQFHj5x7z8Ne1FcQzWFReA4hgoogEFUZelfLUOOr7mARDH2sR0b6fjrGE2DY7Xf8zrf/Y36opOUhEBiWOQSFiclYyjrrz5PDM9JpcFBMf6orlWfO6NLo4UiQODYzB47XN51iLCslUY+nyy0Hrw7sIaCpVuS2QjgDjWJ80EN3MQ3gEBcwiIOoGnGYFRlLTAGz/mjX+rum2Xah3Z6PCcf1Hb9R7azaO5WKBwdGkw53zAEtnNQpGEGoSg3rpxPFe6n88PlVvXo07VdT2wdBZMIXPkoil3nC0gEJzMTHBQE2VIQlO+Nr0Jj2TbFhYUNFhrkyRRLFmi1a/1OxiEPcc0yzqRqpp//2An7IUA1MhW6GmkRCXiifI09SjgLOGgqEuQDxXu1sAgNLVsYA5mowOELaXJlyKahwiUCPbY7MYj3XxFiWDijY+HtJEowtffZn6KKqEE3eTu96L1ATzRCa/9S61ZoWbLczsX3r888qpO866Dto0FdnTRh0T70p6aaDo2jWrMoA2s71pavCpGOabYJUFfbRWFCWM5HcURBoHQrJIzRXC2jQIamPYamBsidBi9XiM4l0NeBNbIFl8BoaOdxEi+fGSoliMzArPFZAiKIrsqFjBE5t59x3XaN1XRoLy/oS+ttvhT2+08NFx8/+00CpDvGKbjoFQplZYa0de6hjRBBLbx/r+HyH4scz4rBewFKt8SkHAoC0qyEcI70KA2SzMhorhpNb22c+MBCdMFGiQspRn5KwcpWonZc0pMFH7BpKRv/ynqTa5y946e5V964tGXdhWf/VBQoM9RvI8tjX2hY5leIp5KnviYw9kXvz/rTfYBJR5wbipLVxo6T/IHRvG+7d47bZccyo42adF9u/bo0OcHjyqt5gePOm6zckW1D/W6/CPVPPJazdpt1TsYV/9DEJA8T0puWB/6L/3U0JlQPzC02Ufl08kjNrIzeRM2Ec+1CZJDphUPxlTbcb1AdDOiXRYENOT0xLqDg1HMXwT09KL3uerUqNJ2vnjkNMmd+duvboU6zmOteTsdD5jn+Q/3FcdzC0RPFg4WL3w86rGUGzkeESaZM8npubFTqxxynsr/R+L83zSi59jvscNToi3Xmzrg6AdGVkwxr+e4CRteUcI5TaIxsZQuD0YyfxFcrV7sRjEfsVgglw+UT+OUelH4IZV5yxUi5hk6V96beVZNZSaCpHocn8TG9etginPetGSv+g6QA9+zzvdsqgSVlZFgfczEx+x879UIqTTRAcujzbceg+eB+wAA1pKkPhmiO+g70LHnGT+3lPhF7fMApgDB1Ayw33nvygjzg1pFOZrf2jVPtMMyolZQsXfO51r7ihqu6wmSttESjK3Xoi/tWsV5JKfj3KsEdcCe/Biwau9zNt3qepuz47r02Hogx1OQ8KIMNZIYyRPN53nRkOJxGIo/B0JMfoCQOZcXLV9s5tBz1C6rtYtwjauVy8bmFlpjZ5YXyU7VMvF+cTLbh/6XZkfLhpldDjmPmWb6Xd4DWGoPmjLL3BY1r5WETCSyZqjbTeFSaTsFwVE9jEm0oSHGQj/d2BGbpj8nIylvACy1e9eVSLkh+bxQl7siZJrJXkUJqLnbiSrTSMTM/KzWZic8rUGqXSlyyHZs6fG0ulbo82zVIqDu3jjJDoL9rFQkbDIDEvKuM8x4RFdRu1NF69JIuJReVWsdAreqAUi5Se42y9SaxcIuzaSx2QzfeiarZ+QEDUwzoVOSMiB6zhI7OXi5MW8pKpTdm/yAKjABqGwPa/OpaD6bqSfzM3yoSd4Mo75mW4X2hsGArXW7XJISaC1JkABZmP9aEocUMc1tyhCwdxZQ1JT7AEubMgI42zcw89V2qg/1K7px6nVquPTZubk72cVi+tKMi5XvWcu+IkDQaAmymetJIB04S6JHa1wE3WOUQTcw245Mx0JBsgWXa5XAojoxvlWeVteLIGpwrxiks2c89qxFNwnxgA1GXakdItWxydctUeksYSPg+UYGn5U0RpDeqIdyPKTdEwYG6ksIY23Ltgr5WnuirY4ofWZf1DOBUwzY8bD3UYqUUx4Syz3PkLMwFX8dLiEdZb/hR/lB/pc49FEc1LZvH4N+FKVTEmH2kj7qyXQ1EgfAeBfG2VE12blgXtXx1O6SB1gtwZo56yyPujpTjggI7z43M+xN9yKZZs1nu3kXgN4JNBvSIMogHrA4ooF/ptP0wHkXzlmxJDXXqTg8d12mWzCp3mHM+uTWwLh4Rn0uo9rAdm9AzrZeXA7ISuHVKIYAvHvhFYzTMrh6+j5IqS1553XnbICuhMsuScplAGtkXnaglkhntg2LfuNRWlTMy1HaAOSomq0A7jRrnkN3rezolzun3aK7GtU+NKj90GyKM33qRc5Z35WVaWz9A+O9Rp0JPIybNUC3SpVIV77aelRaa8Fv27PpbJprtaA+zxOHAfd+a56yY2F+ifPaHwYo70JZyrxWe1Wad47Rp/UC1gcpDM52ryfgu5cwyRy2gTQbMFuCORpk+dJ0V0zj9ZXOERcgvwt5Acm6rmTliNWCZ1OhqVXSHkfGfMsmx6YaSUxTvFNWZYH9top1xpRNsdbQnQYlZlWCsClQqz0SEiHZXjI2L6umu3FoKqtgwzZtWPIkBnRE4EYs1s4zWcl5bpsMgPdvKui1ssGpKzE51bzryN0FvapLD/eVIk3nvTAjoS4ToX3WA8zA+yC7lrKSK6p9qNep4zYrVPPIazVrt1XvHPZ7Bco7ma+shFq1axRRbyz52uJoi5KaUPWWnPCSsGwcmSv5qRYAt+jgqyaEIQKzgnDrLpPtfnUrardjh8Xa1YTIgvUKkxY/Uug9z4PgAfJ+2aacetUt2gTClqslpeUvCiKbXjnbFmir2kOd0gObAlnCOdN47JVvFTlRBZKP85lnxf5PbzQSZ3Z9LjuQA9CDquHpPrlgrt5aYTSdX9S7NS5mFi1h/NRn/nQCZl+ipQRozMDY3xtGM40FttISxJLh2NtMusDumm9kJNV/iq/gpd82tWWguwR+yzotnd8IlUYBZwk/eRNVOOhS9Ayxd+pb7xSiJ/yGsyxMqP0ITnpLlvRO6SYQsOTSTLco7T1gjqUMrQd0C98ewg3Xba/ve5s/vG06lHhysaSxBr4P2ISG48TvlhTf1z5LM3yTKHz5SCxO/SSlKKaBH/bEyZry6fdg9eeYj0UCS3+wv//eh4e4ZeY72d9/47Wf8NqjAfn0zHlAgT/g1OdMPg1AMAlCn082opBEQ05B9OwfDMX8CuChwhd1o6A3CvbTpfP3P7sAr/+E1x/zgP0lbjnyWRhRwF5YdDACyxcDhxNwEF9kcYy3kHAMWXgwBItL8PpPeP0s9v/wU0r88KWfGcGBQ7kLIHIrAIoToEBwLg8WsxA7NYkD1NuIzkqjwjilKFx8/cW2mDABC4E559wREUxDUKMvrluneu3OparrtMUhSg61qe1UG04TqZJZNKQKVVrFwE/6UcxpMB6y+GAYlq4CEipK5+U6DOWbpSgRBT16ioJg8hpFPbnc4enShmsByCkF3dzZZ7nrBcKEJee05O+LVtUIFjrLavn7cD4Ie4VKu8VlQESFheR81WF6UvgqE74KTqoclV679s+aR51a87aK2tsSA7Vw+YgSZgG4rzuP5DmP9VabOt9qX7zDAWCTIIopefWfU0BwKgSdVsOleutLDfyr9bxVBNw4Qpi2hUkSbZTQQMFRkjIC9YNhYd86eZexTu/A/1mC5KFBa5kTOR3nHumTaShyLFMxeRkqCHW9iVmofYfgbCU4573JNXyQ+Yx1pqAp5oJQwykkkZCgliIEIAKUp++GXMgsGuIwxG+uOD+t7tw6Lbx++8YA1Ux58BHThQUzJAwkKuqw5VL9QjJv1Ahahh2QJdkODFC1lrX3tpDzFZGNof+mS9yKctpKyF72NNfwUCXhV5E62SL4MCZXBlvAtHQZqb5rkE0lUWWAXrKnG/uTj9cDdrHpEwRthbLWtlt3/iVk9vc1p9kqppBBHqthqBrB2w4CE8ZiigOO3bBKmklBPWCRGB/1QIzbrKyxR6iVShq96ZKlqzetekX+oVCiCk+kSkNWuuy0w1ORGfChH/Z4XGoAxH/F7upPfPa84t1n/87DK/nfj2KDdvFEV3Lb89HwI6XRMIrj6JWGUSwOGPoo4kF2ZIX4zG/71Pcrl9OI04SxHvX8gIWs8JlHc2A3nnkkfoesSl+5LdC3gn7H9cRWaW6j1qy47Su6cep1arj02bm5o/+6+vWn+8bhXnT1TI052GKh0RAQW4F4PKRk6R1dLR2ZMjsiQyz1DwVY55cD4lYQvxwNV734kUdPzvHedSbSHPpL2Li1IL4wulXol4ydJWwEI38LkGe3EvCk0cYDk45z7FtPTwL25cF+NZ6vpto0jEKx1M8/saGifD23IkC3b/Abzxo9zuK3H1sKky+vyW9JyBa3YSn9OhqqTfBWFsdXwb4jWDptPIyiw5tfa8fnzWEWywBbK9iOpA1yieRWP6/StQPr0rj0pUJt4zJH12u51QB2yZx2Prl6p8B7I1krjRdf661Ij17w9Pkl2e/66fPoqJSefC1yQvkBzhw0a1l7pdqzNdh4Kbum2cHl9KXVJrGByMN9xfFc+uQ0VGL/CcybJbil9Df38XWJxDpnfq2QcUf2bvsbcC8mIBouNVutJnmtzQaPsvxNJe95g0+jbYTrhVqGFaZunZWJ6BPTncIjltv25Jf3jcyRu/SlVnep1qG261QeVVXjKNIsGnCmN7SigYLU8QyapMfIIw5CHyoJ21+BbFB2lX/N+tU70rhCO+qhV342X4ecQRoabIALKEuK57UarXa79Y0cj37N0r2O69Jj60H85FJJqw02Xp6CXZRtaYRS7U3RrjtCeR6zCyWidlhzKfunmiI1SnTbHntap68oYhQEZlThJVW/5CG7UJmBoSFuE/JoMA/U+Ypre70N2uVt2vwK+ordOnAuG8O6eQ5pSSsRcJYoFUGhR1piVk1x9Q2LL3M2tzSxUNDQD6jDkdOVZmJhZceQ5YWB+Y9g7EYyeXwPSvU9uNTCzoB4K4twveG6nuyWtb66FdEg+9KuVZxH8lrN2m3VE32yf6g2cEynlc3JL1oqujf7Q2elRLX7yhTq8pehyIkLaJCXvKV236B7lx4dp0KVWt1pOvoHGBDjS7JP0IpKed5zkxCqUjuj31aujrn2QVSYd8k6LZumCC8UziCCiC1x2m5u7hx5XCm+AbndXtXTMyjRz6TNellgfz/E9PN07Yr7qXDkZ1CiFdrGcU9lho0Eyrxh7zGDiAd4ciTrpeZaPxVAe3uahsGUcmdnBb04toQ5T1nEJupE6WkLEESUcF9mTY1S2HhZJC+YOfuhds1X1O+G/docQVrPyzY3P8+n65n6w9KCHfDwJe1fUT96pSAKX67o559o6IfjlCcfqc+CZ3GPfjSO9yD32/wyGkx/5X0IZfeWICyuBhAnApGpA71aw21LZUjVqX8hp0nV1kN7qhgpAEz+xoDmRGh+vjzFRhZXA4gTgfiFGrXmg+d26G5qKgVw+AUwKIKhG43l08nPyucY8gJAwC2pgUH+kvTpJ0p4Nwp7SQEIFhcBhRNRmJnBEZEBhqDSEKKQr3x4LwCzS4DAiQgs/MkxdgBvpAyIJGWx+GVzZZ2M1r3oNURdZydnPaF2QM6qCQY+BAz2Sof8BcXrB3AbynOntc8idzX6/hfxFkHaStq0+FXlUyw4OxB9VorqgMVD8TbAbtgFYjwkVNdlKep+Pq2o+xnxQm0am8OjcBILMLR1TltNV3ZN4arsJ1KbL0WJZxKTaEAb+Nl3SKisgDH1NMewgfBSmrwUyu9yJLhLvOAxKS7iho4iHOZRDvM4xTJgFIq0gCfpZVH1KS7Bf1mpwC/Q1SsNFoDCRqyICLrAMgDh1J12g2pNupyJlgvggE6S6tzp8iTqHHgonKU4QcOPFqv66g6dvbISHwee6gL+Q7OF/HIE/YGMVqEEYTtfew1GymL/+/KE/jciuSowloLHjJSiMHq9RuSwg8hN66HpycLvoSkyXaFKaLioN8oWPYSRIICUgkaPIZ6yGMlz+pDCkRycoTIwJozFBO1z2eo/TMfYguCpz/yFQfx8jA4BsUKbbAq8iP3JMWBQBv42lz8d015C0NZLhoAHsaY8P2nrNNAgynNZ5FB2gcjGlH776QhRCOgn9ROVG2Qhh+xGjEaGXicl3uk4xPi9VVLwhO4eoFCo51wtu4/ZmwKAqA4d80G+Ywo+vQKqmz7vDrL3zFnaLy0mo4B1ebZTfNdPJ4tN6tOoxyaLTek/Utr3E3rlfMDD3kcS95F/67HJHrRe5ctKKWZ+KF5ajwd+/6DTgeTDSbB6POj7po4aAHYL7LIXkC21OJfRD+klYteH4/cSMQvnMALIZRZ/Cc1lOAdRMGApOxzR6QVA1Saqntu4d9uO99B2xZhD46HxueYcjuFwPHxiK24VmGnG7CVigocbsnGy/xjEHR4UQBnOW7p9HobMz45LOkC1tMBueiXyF+MYPkcx77IklQ/3ErF3MLjzM7jcab4XJxqd9oOBAedWOIMo7MnDI5ePOTsc0WwBG8eDAtV8P3EV2CyNPPBsud3Zp84T5QDi1rCIqFherOqtZqXVFJO6Deeh4zRO85eASzNcvbHwhUcUePJCgGUWLEErHwXWOh8NsExy0cvs5RMLX1gQxbxAfTe/BhSmTVRTPhzxmKXjmMvmkPCCcJpnkzwCr9LjNaNRjghzGzkUAGYo0H1iy8Ht5Nj2CayXDbVD3xdQvjspuwR0+qHrT3o8Zk+st3CVBSQO86vhLS2VBaH/0k9F3v4P8fJG47BIOSA+vloJLBYEimbLgCmlfNjmWxA0lAq6YbGSAJjZruHUaL+AmQ0XCTsrNWZfWm33xul4cvup6mPFbTufnQqSyrLjJrLBYwiT9SwSUGmG6lutXqeaR22n1hQi2NuWQ16rWbuteuR49I+TROqoAAw2TvMc5QU45TOiKKW7ZAkbXZ/iKsFx6Qfu/qHpHqMlQWCzW6PNmjhFGRF0cEpQqqHAPneBydSMTrQ7aEssmSGP1YEI1OyPgWAK5IycqvSpl5kfzSTJp0mYL+E5zasYpKr5OfaF3bCUfjupPM+tAySNqvWEsAQq5zLjNU39F0xYsVFjVA5lcJcLJdFlkYm4Ld0eBDzDbnKOAxp151Kzy00Zpkz0KawmdmWwNxlebEuwTaUB0DOAnjwhZS1NQYpyVpQ0hr7Pqwq4gH2dlX2pG9oBcoYFRPVWpdlqHqkh2rxHDWRE1gqC0ykvJJUWSoK5MxQSIww+ngGC4+HaxlBiq0thPafsugfrM81+nUqhAC1DIW7aU1UxCAKK2YpmFmrZMxrfuW05oEzOanof7Ztz3P2p6CAB5j+s4yai00X5csZGdiEl0ZCL436ikL71WZo4o1FJoXuKepOr/L3F0VNhT/xTGl3Raz8SP5q+kPzvswev1h013E7HuXWp4bpth1pN+lZ1vI5zf09e1fHEj71a85Yara9uhbwWfWnXKs7jPlDFA2dZJeep+C2H0XfeEw+ZibwkzGk0+0jMgLRepF/Fx9hoRDHrjwPqsiCgIacn1h0cjGX+ojyAckkAqBfAlP+V0nd/ELOhuGqGp7gVCweLFzce9VjKD8Z049V5cLNbAl0D6GammYGbYZu3uEMOP4WplgFMGdBoEM2NlAYsFmtzmkRj0e+5PBjI/EUImoaB/CPqh+s4ivs8+wEnX/z2rDc5RIqbc7grV+dRFTcEqLqtM7si5H2Wj6QikKZqjVTcAnAaTWz9D0EgFxYNpzQKX/x+ejCOqxeL9jyipy1ARcHofxhSPA7FET4UFMlrVy9cSml5DyDaiJxKq07ESnOxkoV+qjtYynsAUDOAJsssQgbnFq5OJdeXgEowFj8FxlsgeQdMzwpTOVshOaIBe/LjZQJwiSQqUopui6zyHoDUAOOXMUVLaObD4zvQRGcD5Sj2JzugPBlJuT6QNFp7rrMByqgE1KB2POzVCDz8eSKZrDY+tXBDSGZtdrJXWiNqmiroYJtANYmmtckB/bIineyd0KJSMYBsltmuwSq9aDiee9ELJf4Xaa7JNDfjdZdNTJXbBZ9rKpausfNXJqhc4Gu0IM3yJb01KRIlk5Bm6ZJm+SYyJINdtbyAM9dXy/Ps15Bwnk1cXVWIXel1v1CL6ce16j7OJ17aTvWhvj7xUvvQoPZDsymmXuqO5yopa2C0JiOrNFwDKTAs1my6FPt6SF8kSSaBlNpArZEU0kBzOe9a3zvLebXiC5LQpL1mCBsIp4DVeJKEcZdzVALOLlsTA2a+d42Wv0a37czEZNCSvZkEaYuRKgUUlmkA0Kc+82mbbFfpaBNEZcbrUT09cBSiFuVk6JW+mRgqTXRDnqu1dQrjNQfwuv0aoJFgwyYgztRJa/LBIQRKb383Fc0eGoSTZa0og1j07MdolFauGJ8xv1WgDoYQepUSJL/Y2fMNgOy5//Lon61qM680u6Ibp16nhkufnZs7bIV0Bjj2+YS2qEI15UTIbq2rWDR7YPThDEDsNGueQ3ethcz3zmm3hNa3edeRJ7h4VZce7iuO59J9vaMOXhQ1FtNhrcwh0mPTG5lp9sSoW820z+eYZi30TXYmz/lRZqlopZu01AxVA1JRwGqvc7NyKAVnCT+5gkWqZJ7KV0r8AkDrcgibIzX/H1DAhElypwUA"
open("/content/data/router_train.jsonl", "wb").write(gzip.decompress(base64.b64decode(A)))
open("/content/data/router_train_args.jsonl", "wb").write(gzip.decompress(base64.b64decode(B)))
print(sum(1 for _ in open("/content/data/router_train.jsonl")), "pick examples;",
      sum(1 for _ in open("/content/data/router_train_args.jsonl")), "argument examples")

In [ ]:
#@title 3. Load the data (prompt/completion pairs; the loss is only on the answer)
import json, random
from datasets import Dataset
rows = [json.loads(l) for f in DATA for l in open(f, encoding='utf-8') if l.strip()]
random.Random(1).shuffle(rows)
data = [{"prompt": r["messages"][:2], "completion": r["messages"][2:]} for r in rows]
split = int(len(data) * 0.95)
train_ds, eval_ds = Dataset.from_list(data[:split]), Dataset.from_list(data[split:])
print(len(train_ds), 'train,', len(eval_ds), 'eval')

In [ ]:
#@title 4. Train
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
lora = LoraConfig(r=LORA_RANK, lora_alpha=LORA_RANK*2, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
cfg = SFTConfig(output_dir='/content/run', num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
                per_device_train_batch_size=16, gradient_accumulation_steps=1, logging_steps=20,
                eval_strategy='epoch', save_strategy='no', report_to='none', max_length=512,
                completion_only_loss=True, lr_scheduler_type='cosine', warmup_ratio=0.05,
                bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported())
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=eval_ds,
                     processing_class=tok, peft_config=lora)
trainer.train()

In [ ]:
#@title 5. Quick check on the eval split (exact match of the answer)
model = trainer.model.eval()
ok = 0; n = min(150, len(eval_ds))
for ex in eval_ds.select(range(n)):
    text = tok.apply_chat_template(ex['prompt'], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    out = model.generate(**ids, max_new_tokens=12, do_sample=False)
    got = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    ok += got == ex['completion'][0]['content'].strip()
print(f'{ok}/{n} exact matches on held-back training data (the real test is jitmo-llmbench on testset/heldout.txt)')

In [ ]:
#@title 6. Merge the adapter and save the full model
merged = trainer.model.merge_and_unload()
merged.save_pretrained('/content/merged', safe_serialization=True)
tok.save_pretrained('/content/merged')

In [ ]:
#@title 7. Convert to GGUF and quantise to Q4_K_M (builds llama.cpp on the CPU of this machine, a few minutes)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outfile /content/{NAME}-f16.gguf --outtype f16
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/{NAME}-f16.gguf /content/{NAME}-Q4_K_M.gguf Q4_K_M
!cp /content/{NAME}-Q4_K_M.gguf {OUT_DIR}/
!ls -l {OUT_DIR}
from google.colab import files
files.download(f"{OUT_DIR}/{NAME}-Q4_K_M.gguf")

Done. Download `MyDrive/jitmo/out/<name>-Q4_K_M.gguf` and give the path to the coding agent. It is tested with `jitmo-llmbench` on `testset/heldout.txt`, which this notebook never saw.